Snowflake Functions, Procedures, and Scripting Commands - Learning Notes
*Co-authored with CoCo*

---
# Types of Functions in Snowflake

Snowflake supports four categories of functions you can create:

| Type | Purpose | Created With |
|------|---------|-------------|
| **User-Defined Function (UDF)** | Custom logic for transforms/calculations | `CREATE FUNCTION` |
| **Data Metric Function (DMF)** | Measure data quality on tables/columns | `CREATE DATA METRIC FUNCTION` |
| **Service Function** | Call an endpoint in a Snowpark Container Service | `CREATE FUNCTION ... SERVICE = ...` |
| **External Function** | Call an external API (AWS Lambda, Azure Function, etc.) | `CREATE EXTERNAL FUNCTION` |

> UDFs are covered in detail in Sections 2–2.6 above. Below we cover the other three types.

---

## 1. Data Metric Functions (DMFs)

A DMF measures **data quality** by computing a metric over a table or column. Snowflake runs DMFs automatically on a schedule and stores the results for monitoring.

### What makes DMFs special?
- They are **attached to tables/columns** (not called in queries)
- Snowflake executes them on a schedule and tracks results over time
- Used for data observability: detect nulls, duplicates, freshness issues, out-of-range values
- Results are queryable via `DATA_QUALITY_MONITORING_RESULTS` view

### DMF Signature Rules

A DMF must follow a strict signature:
- Takes exactly one argument: `ARG_T TABLE(...)` — a table reference with specific columns
- Returns exactly one value: a numeric type (`INT`, `FLOAT`, `NUMBER`)

### Creating a DMF

```sql
-- DMF: Count null values in a column
CREATE OR REPLACE DATA METRIC FUNCTION null_count(
    ARG_T TABLE(col VARCHAR)     -- input: a table with one VARCHAR column
)
RETURNS INT
AS
$$
    SELECT COUNT(*) FROM ARG_T WHERE col IS NULL
$$;
```

```sql
-- DMF: Percentage of duplicate values
CREATE OR REPLACE DATA METRIC FUNCTION duplicate_pct(
    ARG_T TABLE(col VARCHAR)
)
RETURNS FLOAT
AS
$$
    SELECT 
        CASE 
            WHEN COUNT(*) = 0 THEN 0.0
            ELSE (COUNT(*) - COUNT(DISTINCT col))::FLOAT / COUNT(*) * 100
        END
    FROM ARG_T
$$;
```

```sql
-- DMF: Check freshness (rows older than 24 hours)
CREATE OR REPLACE DATA METRIC FUNCTION stale_row_count(
    ARG_T TABLE(updated_at TIMESTAMP)
)
RETURNS INT
AS
$$
    SELECT COUNT(*) FROM ARG_T 
    WHERE updated_at < DATEADD(hour, -24, CURRENT_TIMESTAMP())
$$;
```

### Attaching a DMF to a Table

DMFs don't run in queries — they are **attached** to tables and run on a schedule:

```sql
-- Attach DMF to a specific column
ALTER TABLE customers
    SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES'
    ADD DATA METRIC FUNCTION null_count ON (email);

-- Attach to multiple columns
ALTER TABLE orders
    ADD DATA METRIC FUNCTION duplicate_pct ON (order_id);

ALTER TABLE orders
    ADD DATA METRIC FUNCTION stale_row_count ON (updated_at);
```

### Schedule Options

```sql
-- Run when data changes
ALTER TABLE t SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

-- Run on a cron schedule (every 6 hours)
ALTER TABLE t SET DATA_METRIC_SCHEDULE = 'USING CRON 0 */6 * * * UTC';

-- Run every N minutes
ALTER TABLE t SET DATA_METRIC_SCHEDULE = '60 MINUTE';
```

### Viewing DMF Results

```sql
-- Query data quality results
SELECT *
FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
WHERE TABLE_NAME = 'CUSTOMERS'
ORDER BY MEASUREMENT_TIME DESC;
```

### Removing a DMF from a Table

```sql
ALTER TABLE customers
    DROP DATA METRIC FUNCTION null_count ON (email);
```

---

## 2. Service Functions

A service function calls an **endpoint** running inside a **Snowpark Container Services (SPCS)** container. This lets you run custom code (ML models, APIs, complex processing) in Docker containers and expose them as SQL functions.

### How it works:
1. You deploy a container service in SPCS (e.g., a Flask/FastAPI app)
2. The container exposes an HTTP endpoint
3. You create a service function that maps SQL calls to that endpoint
4. Users call it like any other function in SQL

### Creating a Service Function

```sql
-- Service function: calls an ML model running in a container
CREATE OR REPLACE FUNCTION predict_sentiment(text VARCHAR)
RETURNS VARCHAR
SERVICE = my_db.my_schema.sentiment_service    -- the running SPCS service
ENDPOINT = 'predict'                            -- HTTP endpoint name in the service
AS '/predict';                                  -- path on the container
```

```sql
-- Service function with multiple inputs
CREATE OR REPLACE FUNCTION classify_image(image_url VARCHAR, model_name VARCHAR)
RETURNS VARIANT
SERVICE = ml_db.models.image_classifier
ENDPOINT = 'classify'
AS '/classify';
```

### Using a Service Function

```sql
-- Use like any other function in queries
SELECT text, predict_sentiment(text) AS sentiment
FROM customer_reviews;

SELECT url, classify_image(url, 'resnet50') AS classification
FROM product_images;
```

### Key Points

| Aspect | Detail |
|--------|--------|
| **Requires** | A running Snowpark Container Service |
| **Communication** | SQL → HTTP request to container → JSON response back |
| **Use cases** | ML inference, custom algorithms, calling internal microservices |
| **Scaling** | Handled by SPCS compute pool (not warehouse) |
| **Languages** | Any language that can serve HTTP (Python/Flask, Java, Go, etc.) |

### Prerequisites

```sql
-- 1. Create a compute pool (if not exists)
CREATE COMPUTE POOL my_pool
    MIN_NODES = 1 MAX_NODES = 3
    INSTANCE_FAMILY = CPU_X64_S;

-- 2. Create the service (runs your container)
CREATE SERVICE my_db.my_schema.sentiment_service
    IN COMPUTE POOL my_pool
    FROM @my_stage/spec.yaml;   -- service specification file

-- 3. Then create the function that calls it (shown above)
```

---

## 3. External Functions

An external function calls a **remote API outside Snowflake** (e.g., AWS Lambda, Azure Functions, Google Cloud Functions) via an API integration. Snowflake sends data as HTTP POST requests and receives results back.

### How it works:
1. Set up an API Gateway (AWS API Gateway, Azure API Management)
2. Create an API Integration in Snowflake (trust relationship)
3. Create the external function pointing to the gateway URL
4. Snowflake sends rows as JSON batches to the external service

### Step 1: Create an API Integration

```sql
-- API integration establishes trust between Snowflake and the cloud provider
CREATE OR REPLACE API INTEGRATION my_api_integration
    API_PROVIDER = aws_api_gateway           -- or azure_api_management, google_api_gateway
    API_ALLOWED_PREFIXES = ('https://abc123.execute-api.us-east-1.amazonaws.com/prod/')
    ENABLED = TRUE;
```

### Step 2: Create the External Function

```sql
-- External function: calls AWS Lambda for geocoding
CREATE OR REPLACE EXTERNAL FUNCTION geocode_address(address VARCHAR)
RETURNS VARIANT
API_INTEGRATION = my_api_integration
AS 'https://abc123.execute-api.us-east-1.amazonaws.com/prod/geocode';
```

```sql
-- External function: calls a translation service
CREATE OR REPLACE EXTERNAL FUNCTION translate_text(
    text VARCHAR, 
    source_lang VARCHAR, 
    target_lang VARCHAR
)
RETURNS VARCHAR
API_INTEGRATION = my_api_integration
AS 'https://abc123.execute-api.us-east-1.amazonaws.com/prod/translate';
```

### Using External Functions

```sql
-- Use like any SQL function
SELECT address, geocode_address(address) AS coordinates
FROM stores;

SELECT description, translate_text(description, 'en', 'fr') AS french_desc
FROM products;
```

### Request/Response Format

Snowflake sends batches of rows as JSON:

```json
// What Snowflake sends to your API (POST body):
{
  "data": [
    [0, "123 Main St, NYC"],
    [1, "456 Oak Ave, LA"],
    [2, "789 Pine Rd, Chicago"]
  ]
}

// What your API must return:
{
  "data": [
    [0, {"lat": 40.71, "lng": -74.00}],
    [1, {"lat": 34.05, "lng": -118.24}],
    [2, {"lat": 41.88, "lng": -87.63}]
  ]
}
```

**Row format:** `[row_number, arg1, arg2, ...]` — row number must be echoed back.

### Key Points

| Aspect | Detail |
|--------|--------|
| **Requires** | API Integration + cloud API Gateway |
| **Protocols** | HTTPS only (JSON POST requests) |
| **Batch processing** | Snowflake sends rows in batches (not one at a time) |
| **Latency** | Higher than native UDFs (network round-trip) |
| **Cost** | Snowflake compute + external API costs |
| **Security** | Secured via API integration (IAM roles, not API keys in SQL) |
| **Idempotent** | Your API must handle retries (Snowflake may resend batches) |

### External Function vs Service Function

| Comparison | External Function | Service Function |
|------------|------------------|------------------|
| **Where code runs** | Outside Snowflake (AWS/Azure/GCP) | Inside Snowflake (SPCS container) |
| **Network** | Public internet / VPC peering | Snowflake internal network |
| **Latency** | Higher (external network hop) | Lower (internal) |
| **Setup complexity** | API Gateway + IAM + Lambda | Compute pool + container spec |
| **Data governance** | Data leaves Snowflake | Data stays in Snowflake |
| **Best for** | Existing cloud services, third-party APIs | Custom ML models, keeping data in Snowflake |

---

## Summary: Choosing the Right Function Type

| Need | Use |
|------|-----|
| Transform/calculate data in SQL | **UDF** (SQL, Python, Java, JavaScript, Scala) |
| Monitor data quality metrics over time | **Data Metric Function** |
| Run custom ML model / complex logic in a container | **Service Function** (SPCS) |
| Call an existing external API / cloud service | **External Function** |

---
# 1. Overview: Functions vs Procedures

| Feature | Function (UDF) | Stored Procedure |
|---------|---------------|------------------|
| **Purpose** | Compute and return a value | Perform actions (DDL, DML, admin tasks) |
| **Return** | Must return a value (scalar or table) | May or may not return a value |
| **Usage in SQL** | Can be used in SELECT, WHERE, etc. | Called with CALL statement only |
| **Side effects** | Not allowed (no INSERT/UPDATE/DELETE) | Allowed (can modify data, create objects) |
| **Transaction control** | Cannot commit/rollback | Can commit/rollback |
| **Languages** | SQL, Python, Java, JavaScript, Scala | SQL, Python, Java, JavaScript, Scala |

**Key Distinction:** Think of functions as "calculators" (give input, get output) and procedures as "scripts" (perform a sequence of operations).

---
# Handler Syntax Notes

The `HANDLER` clause tells Snowflake which function or class to invoke. Python has two distinct syntaxes depending on where your code lives.

---

## 1. Inline Handler (source code inside the CREATE statement)

Use this when the handler code is written directly between `$$` delimiters.

```sql
CREATE OR REPLACE FUNCTION my_udf(x INT)
RETURNS INT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'compute'       -- just the function name
AS
$$
def compute(x: int) -> int:
    return x * 2
$$;
```

**Syntax:** `HANDLER = '<function_name>'`

- For scalar UDFs/procedures: the handler is the **function name** (e.g., `'compute'`)
- For UDTFs: the handler is the **class name** (e.g., `'MyProcessor'`)
- No module path needed — Snowflake treats the inline code as the module

---

## 2. Stage Handler (source code in a file on a stage)

Use this when the handler code is in a `.py` file uploaded to a Snowflake stage. This is better for larger codebases, version control, and code reuse.

```sql
CREATE OR REPLACE FUNCTION my_udf(x INT)
RETURNS INT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
IMPORTS = ('@my_stage/modules/helpers.py')   -- file(s) on a stage
HANDLER = 'helpers.compute'                  -- module_name.function_name
;
```

**Syntax:** `HANDLER = '<module_name>.<function_name>'`

- `module_name` = the Python file name **without `.py`** (e.g., `helpers.py` → `helpers`)
- `function_name` = the function or class inside that module
- Use `IMPORTS` to specify which stage file(s) to load
- No `AS $$ ... $$` block (there's no inline code)

---

## Side-by-Side Comparison

| Aspect | Inline | Stage |
|--------|--------|-------|
| Code location | Between `$$ ... $$` | `.py` file on a stage |
| `HANDLER` format | `'function_name'` | `'module.function_name'` |
| `IMPORTS` needed? | No | Yes |
| `AS` clause needed? | Yes | No |
| Best for | Small, self-contained logic | Larger modules, shared code |

---
# 2. User-Defined Functions (UDFs)

A UDF is a reusable piece of logic that takes input parameters and returns a result. It can be used anywhere an expression is valid in SQL (SELECT list, WHERE clause, JOIN condition, etc.).

## 2.1 Types of UDFs

| Type | Returns | Use Case |
|------|---------|----------|
| **Scalar UDF** | A single value per input row | Data transformation, calculations |
| **Table UDF (UDTF)** | A set of rows (a table) | Splitting, generating, or expanding data |

## 2.2 The HANDLER Concept

For non-SQL languages (Python, Java, JavaScript, Scala), the `HANDLER` clause is **required**. It tells Snowflake which function or class to execute as the entry point.

**Why is HANDLER needed?**
- The code block (`$$ ... $$`) can contain multiple functions, classes, and helper logic
- Snowflake needs to know exactly which one is the entry point
- SQL UDFs don't need a handler because the entire `$$` block IS the expression

**Handler rules by UDF type:**

| UDF Type | Handler Points To | Example |
|----------|------------------|----------|
| Scalar UDF | A **function** | `HANDLER = 'my_function'` |
| Table UDF (UDTF) | A **class** with `process()` method | `HANDLER = 'MyClass'` |

**Handler rules by code location:**

| Code Location | HANDLER Syntax | Requires |
|---------------|---------------|----------|
| Inline (in `$$ ... $$`) | `'function_name'` | `AS $$ ... $$` block |
| On a stage (`.py` file) | `'module_name.function_name'` | `IMPORTS = (...)` clause |

---
## 2.4 Scalar UDF — SQL Version

A scalar UDF returns one value per row. Think of it like a formula applied to each row.

**No HANDLER clause needed** — the entire `$$` block is the return expression.

```sql
-- Create scalar UDF in SQL
CREATE OR REPLACE FUNCTION celsius_to_fahrenheit(temp_c FLOAT)
RETURNS FLOAT
LANGUAGE SQL
AS
$$
    (temp_c * 9/5) + 32
$$;

-- Using the UDF in a query
SELECT celsius_to_fahrenheit(100) AS boiling_point_f,
       celsius_to_fahrenheit(0) AS freezing_point_f,
       celsius_to_fahrenheit(37) AS body_temp_f;
```

---
## 2.5 Scalar UDF — Python Version (Inline Handler)

Python UDFs let you use Python logic and libraries. The function body goes inside `$$` delimiters.

**Handler rules for Python scalar UDFs:**
- `HANDLER` = the **function name** that Snowflake should call
- The handler function's parameters must match the SQL parameter list (in order and type)
- The handler function's return type must match the SQL `RETURNS` type
- You can define helper functions in the same `$$` block — only the handler is called directly

```sql
-- Create Scalar UDF in Python (inline handler)
CREATE OR REPLACE FUNCTION normalize_name(raw_name VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'normalize'         -- points to the function below
AS
$$
def helper_strip(s: str) -> str:
    """Helper function — NOT the handler, won't be called directly"""
    return s.strip()

def normalize(raw_name: str) -> str:
    """THIS is the handler — Snowflake calls this function"""
    if raw_name is None:
        return None
    return helper_strip(raw_name).title()
$$;

-- Using Python UDF
SELECT normalize_name('  john DOE  ') AS cleaned_name,
       normalize_name('ALICE smith') AS cleaned_name_2;
```

---
## 2.5.1 Scalar UDF — Python Version (Stage Handler)

When your code lives in a `.py` file on a Snowflake stage:

```sql
-- Handler references a function inside a staged file
CREATE OR REPLACE FUNCTION normalize_name(raw_name VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
IMPORTS = ('@my_stage/text_utils.py')   -- load the file from stage
HANDLER = 'text_utils.normalize'        -- module_name.function_name
;
-- NOTE: No AS $$ ... $$ block when using stage handler
```

**Key difference:** The handler becomes `'module_name.function_name'` where `module_name` is the filename without `.py`.

---
## 2.6 Table UDF (UDTF) — Returns Multiple Rows

A UDTF returns a table (multiple rows and columns). Use it when one input row should produce zero, one, or many output rows.

**Common use cases:**
- Splitting a comma-separated string into rows
- Generating a date series
- Expanding JSON arrays into rows

### SQL UDTF

```sql
-- SQL UDTF: generates a sequence of numbers from 1 to N
CREATE OR REPLACE FUNCTION generate_series(n INT)
RETURNS TABLE(num INT)
LANGUAGE SQL
AS
$$
    SELECT SEQ4() + 1 AS num
    FROM TABLE(GENERATOR(ROWCOUNT => n))
$$;

-- Using SQL UDTF: call it with TABLE() in the FROM clause
SELECT num
FROM TABLE(generate_series(5));
```

### Python UDTF — Handler is a CLASS (not a function)

For UDTFs, the `HANDLER` must point to a **class**, not a function. Snowflake calls specific methods on this class:

| Method | When It's Called | Required? |
|--------|-----------------|----------|
| `__init__(self)` | Once per partition (setup) | Optional |
| `process(self, ...)` | Once per input row | **Required** |
| `end_partition(self)` | After all rows in partition | Optional |

**Handler rules for Python UDTFs:**
- `HANDLER = 'ClassName'` — must be the class name, not a method
- `process()` must `yield` tuples matching the `RETURNS TABLE(...)` columns
- Each yielded tuple = one output row
- Return `None` or simply `return` to produce zero rows for an input

```sql
-- Python UDTF: splits a comma-separated string into individual rows
CREATE OR REPLACE FUNCTION split_to_rows(input_str VARCHAR)
RETURNS TABLE(value VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'Splitter'          -- points to the CLASS (not a function)
AS
$$
class Splitter:
    def process(self, input_str: str):
        # Each yield produces one output row
        if input_str is None:
            return
        for item in input_str.split(','):
            yield (item.strip(),)   # tuple must match RETURNS TABLE columns
$$;

-- Using the Python UDTF
SELECT value
FROM TABLE(split_to_rows('apple, banana, cherry, date'));
```

### UDTF with Stage Handler

```sql
CREATE OR REPLACE FUNCTION split_to_rows(input_str VARCHAR)
RETURNS TABLE(value VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
IMPORTS = ('@my_stage/parsers.py')
HANDLER = 'parsers.Splitter'   -- module_name.ClassName
;
```

---
## 2.6 Secure UDFs

A **secure UDF** hides its internal logic from users who can call it but don't own it. This prevents someone from reverse-engineering your business logic.

```sql
CREATE OR REPLACE SECURE FUNCTION calculate_discount(price FLOAT, tier VARCHAR)
RETURNS FLOAT
LANGUAGE SQL
AS
$$
    CASE tier
        WHEN 'gold' THEN price * 0.20
        WHEN 'silver' THEN price * 0.10
        ELSE price * 0.05
    END
$$;
```

**When to use:** When sharing functions via data sharing, or when the logic is proprietary.

---
## 2.7 UDF Overloading

Snowflake supports **function overloading** — multiple functions with the same name but different parameter signatures.

```sql
-- These are TWO different functions despite the same name:
CREATE FUNCTION add_values(a INT, b INT) RETURNS INT AS $$ a + b $$;
CREATE FUNCTION add_values(a FLOAT, b FLOAT, c FLOAT) RETURNS FLOAT AS $$ a + b + c $$;
```

Snowflake resolves which version to call based on the arguments you pass.

---
# 3. Stored Procedures

A stored procedure is a named block of code that performs actions. Unlike functions, procedures **can modify data** (INSERT, UPDATE, DELETE), **execute DDL** (CREATE TABLE, DROP), and **control transactions**.

A procedure can be written in one of the following languages:
1. Java (using Snowpark)
2. JavaScript
3. Python (using Snowpark)
4. Scala (using Snowpark)
5. Snowflake Scripting

## 3.1 When to Use Procedures (not Functions)

- ETL/ELT pipelines (load, transform, move data)
- Administrative tasks (grant roles, create users)
- Multi-step operations that need transaction control
- Operations requiring dynamic SQL
- Tasks that modify database state

Note:-
SQLROWCOUNT is a Snowflake Scripting global variable that stores the number of rows affected by the most recently executed DML statement (such as INSERT, UPDATE, DELETE, or MERGE). It is commonly used inside Snowflake stored procedures and scripting blocks.

---
## 3.2 Stored Procedure — SQL Version

SQL procedures use **Snowflake Scripting** (covered in Section 4). The body is enclosed in `BEGIN...END`.

**Handler:** SQL procedures do NOT use a `HANDLER` clause. The `BEGIN...END` block is the entry point.

**Important:** Inside the procedure body, variables and parameters must be prefixed with `:` when used in SQL statements (e.g., `:my_var`). Without the colon, Snowflake treats them as column names.

```sql
-- SQL Stored Procedure: archives old orders and returns count
CREATE OR REPLACE PROCEDURE archive_old_orders(cutoff_date DATE)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    LET row_count INT := 0;

    -- Insert old orders into archive table
    INSERT INTO orders_archive
    SELECT * FROM orders WHERE order_date < :cutoff_date;

    -- Get count of moved rows
    row_count := SQLROWCOUNT;

    -- Delete from source
    DELETE FROM orders WHERE order_date < :cutoff_date;

    RETURN 'Archived ' || :row_count || ' orders';
END;
$$;
```

**Calling a stored procedure** (always use `CALL`):

```sql
CALL archive_old_orders('2023-01-01');
```

---
## 3.3 Stored Procedure — Python Version

Python procedures get a `session` object (Snowpark Session) as their first parameter, which lets them execute SQL, read tables, and write data.

**Handler rules for Python procedures:**
- `HANDLER` = the **function name** to call (same as UDFs)
- The handler function's **first parameter** is always `session` (Snowpark Session) — you do NOT declare it in the SQL parameter list
- Remaining parameters map to the SQL parameters in order
- For in-line handlers: `HANDLER = 'function_name'`
- For stage handlers: `HANDLER = 'module_name.function_name'` + `IMPORTS = (...)`

**Key differences from Python UDFs:**
- First parameter is always the Snowpark `Session` (invisible in SQL signature)
- Can execute any SQL via `session.sql()`
- Can modify data and perform DDL
- Called with `CALL`, not usable in SELECT

### Python Stored Procedure (inline handler)
```sql
CREATE OR REPLACE PROCEDURE create_summary(source_table VARCHAR, target_table VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'               -- points to the function below
AS
$$
def run(session, source_table: str, target_table: str) -> str:
    # 'session' is injected by Snowflake — not in SQL param list
    df = session.table(source_table)
    row_count = df.count()
    df.write.mode('overwrite').save_as_table(target_table)
    return f'Created {target_table} with {row_count} rows from {source_table}'
$$;
```

### Python Stored Procedure — Stage Handler

```sql
-- Handler in a staged file
CREATE OR REPLACE PROCEDURE create_summary(source_table VARCHAR, target_table VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
IMPORTS = ('@my_stage/etl_logic.py')
HANDLER = 'etl_logic.run'     -- module_name.function_name
;
```

---
## 3.4 Caller's Rights vs Owner's Rights

This controls **whose privileges** are used when the procedure runs.

| Mode | Keyword | Behavior |
|------|---------|----------|
| **Owner's Rights** (default) | `EXECUTE AS OWNER` | Runs with the privileges of the procedure's owner |
| **Caller's Rights** | `EXECUTE AS CALLER` | Runs with the privileges of the user who calls it |

**When to use Caller's Rights:**
- When the procedure should operate on tables the *caller* has access to
- When you don't want to grant the caller extra privileges through the procedure

**When to use Owner's Rights:**
- When the procedure needs access to objects the caller shouldn't directly access
- When you want to encapsulate and control access

```sql
CREATE OR REPLACE PROCEDURE my_proc()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER  -- or EXECUTE AS OWNER (default)
AS $$ BEGIN RETURN 'done'; END; $$;
```

---
## 3.5 Anonymous Procedures (Inline, No Stored Definition)

If you need to run procedural logic once without saving it as a named object, use an **anonymous procedure** with `WITH ... CALL`. It is necessary to use `CALL` keyword during definition.

Useful for: ad-hoc scripts, migrations, one-time fixes.

```sql
-- Anonymous procedure: runs immediately, not stored in the database
WITH my_temp_proc AS PROCEDURE()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    RETURN 'This ran without being stored anywhere: ' || CURRENT_TIMESTAMP()::VARCHAR;
END;
$$
CALL my_temp_proc();
```

---
# 4. Snowflake Scripting (SQL Procedural Language)

Snowflake Scripting extends SQL with programming constructs (variables, loops, conditionals, cursors, exceptions). It is used inside **SQL stored procedures** and **anonymous blocks**.

## 4.1 Variables

Variables are declared with `LET` or in a `DECLARE` block. They hold values during procedure execution.

```sql
DECLARE
    my_name VARCHAR DEFAULT 'World';
    counter INT DEFAULT 0;
BEGIN
    LET greeting VARCHAR := 'Hello, ' || my_name;
    RETURN greeting;
END;
```

**Rules:**
- Use `:variable_name` when referencing a variable inside a SQL statement
- Use `variable_name` (no colon) in control flow (IF, WHILE, assignments)
- `:=` is the assignment operator

---
## 4.2 Conditional Logic (IF / CASE)

```sql
BEGIN
    LET status VARCHAR;
    LET score INT := 85;

    IF (score >= 90) THEN
        status := 'Excellent';
    ELSEIF (score >= 70) THEN
        status := 'Good';
    ELSE
        status := 'Needs Improvement';
    END IF;

    RETURN status;  -- Returns 'Good'
END;
```

**Note:** This is different from the SQL `CASE` expression. `IF/ELSEIF/ELSE` is for control flow in scripting. `CASE` is for expressions within SQL queries.

---
## 4.3 Loops

Snowflake Scripting supports four loop types:

### FOR Loop (counter-based)
```sql
BEGIN
    LET total INT := 0;
    FOR i IN 1 TO 10 DO
        total := total + i;
    END FOR;
    RETURN total;  -- Returns 55
END;
```

### WHILE Loop
```sql
BEGIN
    LET counter INT := 1;
    LET result VARCHAR := '';
    WHILE (counter <= 5) DO
        result := result || counter::VARCHAR || ' ';
        counter := counter + 1;
    END WHILE;
    RETURN result;  -- Returns '1 2 3 4 5 '
END;
```

### LOOP (infinite until BREAK)
```sql
BEGIN
    LET x INT := 0;
    LOOP
        x := x + 1;
        IF (x > 3) THEN
            BREAK;
        END IF;
    END LOOP;
    RETURN x;  -- Returns 4
END;
```

### REPEAT...UNTIL (do-while equivalent)
```sql
BEGIN
    LET counter INT := 0;
    REPEAT
        counter := counter + 1;
    UNTIL (counter >= 5)
    END REPEAT;
    RETURN counter;  -- Returns 5
END;
```

---
## 4.4 Cursors — Iterating Over Query Results

A **cursor** lets you process query results row by row. Useful when you need different actions depending on each row's values.

```sql
DECLARE
    result VARCHAR DEFAULT '';
    c1 CURSOR FOR SELECT name, age FROM employees WHERE department = 'Engineering';
BEGIN
    FOR record IN c1 DO
        result := result || record.name || ' (' || record.age || '), ';
    END FOR;
    RETURN result;
END;
```

**Key points:**
- `CURSOR FOR <query>` defines what data to iterate over
- `FOR record IN cursor_name DO` loops through each row
- Access columns with dot notation: `record.column_name`
- Cursors are forward-only (you can't go back to a previous row)

---
## 4.5 RESULTSET — Dynamic SQL

A `RESULTSET` holds the result of a dynamically constructed SQL statement. Use this when you need to build SQL at runtime.

```sql
DECLARE
    query VARCHAR;
    res RESULTSET;
BEGIN
    query := 'SELECT COUNT(*) AS cnt FROM ' || :table_name;
    res := (EXECUTE IMMEDIATE :query);
    RETURN TABLE(res);
END;
```

**Use cases:**
- Table name is a parameter (can't use bind variables for identifiers)
- Building WHERE clauses dynamically
- Generic utility procedures that work on any table

---
## 4.6 EXECUTE IMMEDIATE — Dynamic SQL Execution

`EXECUTE IMMEDIATE` compiles and runs a SQL string at runtime. This is how you do **dynamic SQL** in Snowflake Scripting — when table names, column names, or entire queries are built dynamically.

---

### Syntax 1: EXECUTE IMMEDIATE (basic)

Runs a SQL string stored in a variable or passed as a literal.

```sql
EXECUTE IMMEDIATE '<sql_string>';
-- or
EXECUTE IMMEDIATE :variable_containing_sql;
```

**Example — Dynamic table operations:**

```sql
CREATE OR REPLACE PROCEDURE drop_table_if_old(table_name VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    -- Build the SQL string dynamically
    LET sql_stmt VARCHAR := 'DROP TABLE IF EXISTS ' || :table_name;
    
    -- Execute it
    EXECUTE IMMEDIATE :sql_stmt;
    
    RETURN 'Dropped: ' || :table_name;
END;
$$;
```

---

### Syntax 2: EXECUTE IMMEDIATE ... INTO (capture scalar result)

Captures a single-row result into variable(s).

```sql
EXECUTE IMMEDIATE :query INTO :variable;
```

**Example — Get row count from a dynamic table name:**

```sql
CREATE OR REPLACE PROCEDURE get_row_count(table_name VARCHAR)
RETURNS INT
LANGUAGE SQL
AS
$$
DECLARE
    row_count INT;
    query VARCHAR;
BEGIN
    query := 'SELECT COUNT(*) FROM ' || :table_name;
    
    -- Execute and capture the result into row_count
    EXECUTE IMMEDIATE :query INTO :row_count;
    
    RETURN row_count;
END;
$$;
```

**Rules for INTO:**
- The query must return exactly **one row**
- Number of variables must match number of columns
- Multiple columns: `EXECUTE IMMEDIATE :q INTO :var1, :var2, :var3;`

---

### Syntax 3: EXECUTE IMMEDIATE ... USING (bind parameters)

Passes bind values into placeholders (`?`) in the SQL string. This is safer than string concatenation for **values** (not identifiers).

```sql
EXECUTE IMMEDIATE :query USING (value1, value2, ...);
```

**Example — Safe parameterized query:**

```sql
CREATE OR REPLACE PROCEDURE insert_log_entry(msg VARCHAR, severity INT)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    LET sql_stmt VARCHAR := 'INSERT INTO audit_log(message, severity, ts) VALUES (?, ?, CURRENT_TIMESTAMP())';
    
    -- ? placeholders are replaced by USING values in order
    EXECUTE IMMEDIATE :sql_stmt USING (:msg, :severity);
    
    RETURN 'Logged: ' || :msg;
END;
$$;
```

**Important:** `USING` binds **values** only (like WHERE conditions, INSERT values). It **cannot** bind identifiers (table names, column names) — use string concatenation for those.

---

### Syntax 4: Combining INTO and USING together

```sql
EXECUTE IMMEDIATE :query INTO :result USING (bind_values);
```

**Example — Parameterized lookup:**

```sql
CREATE OR REPLACE PROCEDURE lookup_customer_name(cust_id INT)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    cust_name VARCHAR;
    query VARCHAR := 'SELECT name FROM customers WHERE id = ?';
BEGIN
    EXECUTE IMMEDIATE :query INTO :cust_name USING (:cust_id);
    RETURN :cust_name;
END;
$$;
```

---

### Syntax 5: EXECUTE IMMEDIATE with RESULTSET

When the dynamic query returns **multiple rows**, capture it in a `RESULTSET` and return as a table.

```sql
CREATE OR REPLACE PROCEDURE search_table(table_name VARCHAR, col VARCHAR, search_val VARCHAR)
RETURNS TABLE()
LANGUAGE SQL
AS
$$
DECLARE
    query VARCHAR;
    res RESULTSET;
BEGIN
    -- Identifiers via concatenation, values via USING
    query := 'SELECT * FROM ' || :table_name || ' WHERE ' || :col || ' = ?';
    
    res := (EXECUTE IMMEDIATE :query USING (:search_val));
    
    RETURN TABLE(res);
END;
$$;
```

---

### Quick Reference

| Syntax | Purpose | Returns |
|--------|---------|--------|
| `EXECUTE IMMEDIATE :sql` | Run dynamic SQL (DDL/DML) | Nothing (side effect) |
| `EXECUTE IMMEDIATE :sql INTO :var` | Run query, capture one row | Scalar value(s) |
| `EXECUTE IMMEDIATE :sql USING (:v1, :v2)` | Bind values safely (avoids SQL injection) | Nothing (side effect) |
| `EXECUTE IMMEDIATE :sql INTO :var USING (...)` | Bind values + capture result | Scalar value(s) |
| `res := (EXECUTE IMMEDIATE :sql)` | Capture multi-row result in RESULTSET | Table via `RETURN TABLE(res)` |

---

### When to Use What?

| Scenario | Approach |
|----------|----------|
| Dynamic **table/column names** | String concatenation: `'SELECT * FROM ' \|\| :tbl` |
| Dynamic **filter values** | `USING`: `'WHERE id = ?'` + `USING (:id)` |
| Both dynamic identifiers and values | Concatenate identifiers + USING for values |
| Need result of dynamic query | `INTO` for one row, `RESULTSET` for many rows |

**Security note:** Always use `USING` for user-supplied values to prevent SQL injection. Only use concatenation for identifiers (table/column names) that you've validated.

---
## 4.7 Exception Handling

Handle errors gracefully so your procedure doesn't crash on unexpected conditions.

```sql
DECLARE
    my_exception EXCEPTION (-20001, 'Custom error occurred');
BEGIN
    BEGIN
        INSERT INTO target_table SELECT * FROM source_table;
    EXCEPTION
        WHEN STATEMENT_ERROR THEN
            RETURN 'SQL Error: ' || SQLERRM;
        WHEN OTHER THEN
            RETURN 'Unexpected error: ' || SQLERRM;
    END;
    RETURN 'Success';
END;
```

**Built-in exception types:**
- `STATEMENT_ERROR` — a SQL statement failed
- `EXPRESSION_ERROR` — an expression evaluation failed
- `OTHER` — catches everything else (use as a fallback)

**Custom exceptions:** Declare with a number and message, raise with `RAISE`:
```sql
DECLARE
    invalid_input EXCEPTION (-20002, 'Input validation failed');
BEGIN
    IF (:param IS NULL) THEN
        RAISE invalid_input;
    END IF;
END;
```

In [ ]:
%%sql -r create_scripting_proc
-- Complete Snowflake Scripting example: uses variables, loops,
-- conditionals, and exception handling
CREATE OR REPLACE PROCEDURE process_batch(batch_size INT)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    processed INT DEFAULT 0;
    errors INT DEFAULT 0;
    msg VARCHAR DEFAULT '';
BEGIN
    IF (:batch_size <= 0) THEN
        RETURN 'Error: batch_size must be positive';
    END IF;

    FOR i IN 1 TO :batch_size DO
        BEGIN
            processed := processed + 1;
        EXCEPTION
            WHEN OTHER THEN
                errors := errors + 1;
        END;
    END FOR;

    msg := 'Processed: ' || :processed || ', Errors: ' || :errors;
    RETURN msg;
END;
$$;

---
# 5. Snowpark Python — Procedural Logic in Python

Snowpark is Snowflake's Python (also Java/Scala) library that lets you write data transformations using DataFrame APIs, similar to PySpark or Pandas.

## 5.1 When to Use Snowpark vs SQL Scripting

| Use SQL Scripting When... | Use Snowpark (Python) When... |
|--------------------------|-------------------------------|
| Logic is mainly SQL with light control flow | Logic is complex (ML, string parsing, APIs) |
| You're comfortable with SQL | You prefer Python idioms |
| Performance is critical (fewer handoffs) | You need Python libraries (pandas, sklearn) |
| Simple variable substitution and loops | Complex data transformations |

---
## 5.2 Snowpark DataFrame Basics (Inside a Procedure)

Inside a Python procedure, you use the `session` object to interact with Snowflake:

```python
def run(session):
    # Read a table as a DataFrame
    df = session.table('my_table')
    
    # Filter rows (like WHERE)
    filtered = df.filter(df['status'] == 'active')
    
    # Select columns
    selected = filtered.select('id', 'name', 'amount')
    
    # Aggregate (like GROUP BY)
    summary = df.group_by('category').agg(
        sum('amount').alias('total'),
        count('id').alias('num_records')
    )
    
    # Write results
    summary.write.mode('overwrite').save_as_table('summary_output')
    
    return 'Done'
```

**Key Snowpark concepts:**
- Operations are **lazy** — nothing executes until you call `.collect()`, `.show()`, or `.save_as_table()`
- This pushes computation to Snowflake's engine (not your local machine)
- The DataFrame API mirrors SQL semantics: filter=WHERE, select=SELECT, group_by=GROUP BY

---
# 6. Managing Functions and Procedures

## 6.1 Listing, Describing, and Dropping

```sql
-- List all UDFs in current schema
SHOW USER FUNCTIONS;

-- List all procedures in current schema
SHOW PROCEDURES;

-- Get details of a specific function (including body)
DESCRIBE FUNCTION celsius_to_fahrenheit(FLOAT);

-- Get details of a specific procedure
DESCRIBE PROCEDURE archive_old_orders(DATE);

-- Drop a function (must include parameter types for overloaded names)
DROP FUNCTION celsius_to_fahrenheit(FLOAT);

-- Drop a procedure
DROP PROCEDURE archive_old_orders(DATE);
```

**Note:** You must specify the argument types in `DESCRIBE` and `DROP` because Snowflake allows overloading (same name, different signatures).

## 6.2 Granting Privileges

```sql
-- Allow a role to use a function
GRANT USAGE ON FUNCTION celsius_to_fahrenheit(FLOAT) TO ROLE analyst_role;

-- Allow a role to call a procedure
GRANT USAGE ON PROCEDURE archive_old_orders(DATE) TO ROLE etl_role;
```

---
# 7. Quick Reference: Type Mappings

## SQL to Python Type Mapping

| SQL Type | Python Type |
|----------|------------|
| `INT`, `BIGINT` | `int` |
| `FLOAT`, `DOUBLE` | `float` |
| `VARCHAR`, `STRING` | `str` |
| `BOOLEAN` | `bool` |
| `DATE` | `datetime.date` |
| `TIMESTAMP` | `datetime.datetime` |
| `VARIANT` | `dict` or `list` |
| `ARRAY` | `list` |
| `OBJECT` | `dict` |

---
# 8. Common Patterns and Best Practices

## 8.1 Pattern: Dynamic Table Processing

```sql
CREATE OR REPLACE PROCEDURE count_rows(table_name VARCHAR)
RETURNS INT
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    row_count INT;
    query VARCHAR;
BEGIN
    query := 'SELECT COUNT(*) FROM ' || :table_name;
    EXECUTE IMMEDIATE :query INTO :row_count;
    RETURN row_count;
END;
$$;
```

## 8.2 Pattern: Error Logging

```sql
CREATE OR REPLACE PROCEDURE safe_load(src VARCHAR, tgt VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    BEGIN
        EXECUTE IMMEDIATE 'INSERT INTO ' || :tgt || ' SELECT * FROM ' || :src;
        RETURN 'Success: loaded ' || SQLROWCOUNT || ' rows';
    EXCEPTION
        WHEN OTHER THEN
            INSERT INTO error_log(error_time, error_msg)
                VALUES (CURRENT_TIMESTAMP(), :SQLERRM);
            RETURN 'Failed: ' || :SQLERRM;
    END;
END;
$$;
```

## 8.3 Best Practices Summary

1. **Use functions** for calculations/transforms that return values; **use procedures** for actions that modify state
2. **Prefer SQL UDFs** for simple expressions (best performance, no language overhead)
3. **Use Python** when you need libraries, complex logic, or are more comfortable with it
4. **Always specify `RUNTIME_VERSION`** for Python/Java to avoid surprises on version changes
5. **Use `:variable`** prefix in SQL statements inside procedures (the #1 beginner mistake)
6. **Use `EXECUTE AS CALLER`** when the procedure should use the caller's permissions
7. **Handle exceptions** in procedures to avoid cryptic errors for end users
8. **Use `PACKAGES = (...)`** to declare Python dependencies explicitly

---
# Snowpark Session: UDFs vs Procedures

## The Core Question

> Can we use Snowpark in both Python UDFs and Python Procedures? Where do we get a session and where don't we?

---

## Answer: Session Availability

| Object Type | Gets Snowpark Session? | Can Use Snowpark APIs? | Can Run SQL? |
|-------------|----------------------|----------------------|-------------|
| **Python Stored Procedure** | **YES** — `session` is the first parameter | Full Snowpark (DataFrames, `.sql()`, `.table()`, write ops) | Yes |
| **Python UDF (Scalar)** | **NO** — no session available | Cannot use Snowpark at all | No |
| **Python UDTF (Table)** | **NO** — no session available | Cannot use Snowpark at all | No |

---

## Why?

- **UDFs** are designed to be **pure functions** — they take input, return output, with no side effects. They run in a sandboxed environment where each row is processed independently. Giving them a session would allow data modification, breaking the function contract.
- **Procedures** are designed to **perform actions** — they need a session to interact with the database (query tables, insert data, create objects, etc.).

---

## Procedure: Gets Session (Implicit First Parameter)

```sql
CREATE OR REPLACE PROCEDURE my_proc(table_name VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')   -- REQUIRED for session
HANDLER = 'run'
AS
$$
def run(session, table_name: str) -> str:
    # 'session' is injected by Snowflake automatically
    # It is NOT declared in the SQL parameter list
    df = session.table(table_name)          # read a table
    count = df.count()                       # use Snowpark DataFrame API
    session.sql("INSERT INTO log VALUES (CURRENT_TIMESTAMP())").collect()  # run SQL
    return f'{table_name} has {count} rows'
$$;

CALL my_proc('my_database.my_schema.customers');
```

**Key points:**
- Must include `PACKAGES = ('snowflake-snowpark-python')`
- `session` is always the **first** parameter in the Python function
- `session` is **NOT** listed in the SQL `CREATE PROCEDURE` parameter list
- The session has the same privileges as the procedure's execution context (caller's or owner's rights)

---

## UDF: No Session — Pure Python Only

```sql
CREATE OR REPLACE FUNCTION clean_email(email VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'clean'
AS
$$
def clean(email: str) -> str:
    # NO session parameter — cannot access Snowflake here
    # Can only use pure Python logic and imported packages
    if email is None:
        return None
    return email.strip().lower()
$$;

-- Used inline in queries (NOT with CALL)
SELECT clean_email(email) FROM customers;
```

**What you CAN do in a Python UDF (without session):**
- Pure Python logic (string manipulation, math, regex)
- Use third-party libraries (`PACKAGES = ('pandas', 'numpy', 'scikit-learn', ...)`)
- Read files imported from a stage (`IMPORTS = (...)`) — static files only, not querying tables

**What you CANNOT do in a Python UDF:**
- Query other tables
- Insert/update/delete data
- Execute SQL statements
- Access the Snowpark session in any way

---

## Common Mistake: Trying to Use Session in a UDF

```sql
-- THIS WILL FAIL at runtime
CREATE OR REPLACE FUNCTION bad_udf(id INT)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'lookup'
AS
$$
def lookup(session, id: int) -> str:    # WRONG — UDFs don't get session
    result = session.sql(f"SELECT name FROM users WHERE id = {id}").collect()
    return result[0][0]
$$;
```

**Fix:** If you need to look up data from another table, use a **JOIN** in your SQL query instead of trying to query inside the UDF. Or convert it to a **procedure** if the logic truly requires database access.

---

## When to Use What?

| Need | Use | Why |
|------|-----|-----|
| Transform a column value (row by row) | **UDF** | Pure function, no DB access needed |
| Use pandas/numpy/ML on input data | **UDF** with `PACKAGES` | Libraries available without session |
| Read/write tables, run SQL, DDL | **Procedure** | Needs session for DB interaction |
| Multi-step ETL pipeline | **Procedure** | Needs session + transaction control |
| Callable from SELECT/WHERE/JOIN | **UDF** | Procedures can't be used in queries |
| Callable only with CALL | **Procedure** | Procedures are action-oriented |

---

## Summary Rule

> **Procedure = gets `session` = can do anything in Snowflake**
>
> **UDF = no `session` = pure computation only, but usable inside SQL queries**